In [ ]:
import pickle
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt

# Hàm ensemble và hiển thị feature importance cho từng mô hình
def ensemble_and_display_importance(X, y, rf_model_path, gb_model_path, stacking_model_path, test_size=0.3, random_state=42):
    # Chia dữ liệu thành train và test
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=random_state)
    
    # Load các mô hình đã huấn luyện
    with open(rf_model_path, 'rb') as f:
        rf_model = pickle.load(f)
    
    with open(gb_model_path, 'rb') as f:
        gb_model = pickle.load(f)
    
    with open(stacking_model_path, 'rb') as f:
        stacking_model = pickle.load(f)
    
    # Ensemble Model: Sử dụng Stacking để kết hợp các mô hình (Random Forest, Gradient Boosting)
    # Meta-model cho Stacking
    meta_model = LogisticRegression()
    
    # Stacking ensemble model
    stacking_model_ensemble = StackingClassifier(
        estimators=[('rf', rf_model), ('gb', gb_model)],
        final_estimator=meta_model
    )
    
    # Huấn luyện ensemble model
    stacking_model_ensemble.fit(X_train, y_train)
    
    # Dự đoán và tính toán độ chính xác
    y_pred = stacking_model_ensemble.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    print(f"Ensemble Model Accuracy: {accuracy * 100:.2f}%")
    
    # Hiển thị Feature Importance cho từng mô hình
    # 1. Random Forest Importance
    if hasattr(rf_model, 'feature_importances_'):
        rf_importance = rf_model.feature_importances_
        print("Random Forest Feature Importance:")
        print(rf_importance)
        plot_feature_importance(rf_importance, 'Random Forest')
    
    # 2. Gradient Boosting Importance
    if hasattr(gb_model, 'feature_importances_'):
        gb_importance = gb_model.feature_importances_
        print("Gradient Boosting Feature Importance:")
        print(gb_importance)
        plot_feature_importance(gb_importance, 'Gradient Boosting')
    
    # 3. Stacking Model Feature Importance (Meta-model)
    # Stacking meta-model chỉ có trọng số của các mô hình con
    print("Stacking Meta-model Importance (weights of RF and GB):")
    meta_model_weights = stacking_model_ensemble.final_estimator_.coef_
    print(meta_model_weights)
    
# Hàm vẽ biểu đồ Feature Importance
def plot_feature_importance(importances, model_name):
    # Vẽ biểu đồ Feature Importance
    plt.figure(figsize=(10, 6))
    plt.barh(np.arange(len(importances)), importances)
    plt.yticks(np.arange(len(importances)), [f'Feature {i+1}' for i in range(len(importances))])
    plt.xlabel('Importance')
    plt.title(f'{model_name} - Feature Importance')
    plt.show()

# Ví dụ sử dụng
if __name__ == "__main__":
    # Load dữ liệu Iris
    data = load_iris()
    X = data.data
    y = data.target

    # Đường dẫn tới các mô hình đã được lưu
    rf_model_path = 'rf_model.pkl'  # Đường dẫn tới mô hình Random Forest đã được lưu
    gb_model_path = 'gb_model.pkl'  # Đường dẫn tới mô hình Gradient Boosting đã được lưu
    stacking_model_path = 'stacking_model.pkl'  # Đường dẫn tới mô hình Stacking đã được lưu
    
    # Gọi hàm ensemble và hiển thị feature importance
    ensemble_and_display_importance(X, y, rf_model_path, gb_model_path, stacking_model_path)
Giải thích:
ensemble_and_display_importance:

Hàm này nhận vào các tham số:

X, y: Dữ liệu huấn luyện.

rf_model_path, gb_model_path, stacking_model_path: Đường dẫn tới các mô hình đã được huấn luyện trước đó và lưu vào file pickle.

test_size và random_state: Để chia dữ liệu thành train và test.

Hàm này sẽ tải các mô hình đã huấn luyện, kết hợp chúng lại với nhau bằng cách sử dụng Stacking, và tính toán độ chính xác trên tập test.

Hàm cũng sẽ hiển thị feature importance cho từng mô hình (Random Forest và Gradient Boosting), và trọng số của các mô hình con trong Stacking.

plot_feature_importance:

Hàm này nhận vào một mảng importances và tên của mô hình, sau đó vẽ biểu đồ feature importance của mô hình đó.

Cách sử dụng:

Trước khi chạy, bạn cần huấn luyện các mô hình RandomForestClassifier, GradientBoostingClassifier, và StackingClassifier và lưu chúng vào các file pickle. Sau đó, bạn có thể gọi hàm ensemble_and_display_importance với các đường dẫn đến các mô hình đã lưu.

Cách huấn luyện và lưu mô hình:
Nếu bạn chưa lưu các mô hình trước đó, bạn có thể huấn luyện chúng và lưu lại như sau:

python
Copy code
# Huấn luyện mô hình Random Forest
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# Lưu mô hình Random Forest
with open('rf_model.pkl', 'wb') as f:
    pickle.dump(rf_model, f)

# Huấn luyện mô hình Gradient Boosting
gb_model = GradientBoostingClassifier(n_estimators=100, random_state=42)
gb_model.fit(X_train, y_train)

# Lưu mô hình Gradient Boosting
with open('gb_model.pkl', 'wb') as f:
    pickle.dump(gb_model, f)

# Huấn luyện mô hình Stacking
meta_model = LogisticRegression()
stacking_model_ensemble = StackingClassifier(
    estimators=[('rf', rf_model), ('gb', gb_model)],
    final_estimator=meta_model
)
stacking_model_ensemble.fit(X_train, y_train)

# Lưu mô hình Stacking
with open('stacking_model.pkl', 'wb') as f:
    pickle.dump(stacking_model_ensemble, f)